In [ ]:
# VARIANT 1: 5-seed CatBoost + state_crosses features
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, random
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier

TARGET, ID_COL, FOLDS = 'Subscribed', 'id', 5
SEEDS = [42, 2024, 3407, 777, 1337]  # 5 seeds for more stability
MONTH_MAP = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}

CB_PARAMS = {
    'iterations': 4000, 'learning_rate': 0.012, 'depth': 6,
    'l2_leaf_reg': 4.5, 'random_strength': 1.2, 'bagging_temperature': 0.8,
    'bootstrap_type': 'Bayesian', 'loss_function': 'Logloss', 'eval_metric': 'AUC',
    'allow_writing_files': False, 'verbose': False, 'thread_count': -1
}

BASE_CAT = ['job','marital','education','default','housing','loan','contact','month','poutcome']
def rank_norm(v): return pd.Series(v).rank(method='average', pct=True).to_numpy()
def _s(d,c,l=False): v=d[c].fillna('missing').astype(str); return v.str.lower() if l else v

print('Variant 1: 5-seed CatBoost with state_crosses features')

In [ ]:
def build_state_crosses(data):
    df = data.copy()
    if ID_COL in df.columns: df = df.drop(columns=[ID_COL])
    
    m,j,ma,e,ct,po,l,d,h = [_s(df,c,c=='month') for c in ['month','job','marital','education','contact','poutcome','loan','default','housing']]
    
    df['month'] = m
    df['month_num'] = m.map(MONTH_MAP).fillna(0).astype(np.int16)
    df['pdays_was_missing'] = (df['pdays']==-1).astype(np.int8)
    df['pdays_clean'] = df['pdays'].replace(-1, 999)
    
    for col in ['duration','balance','campaign','previous']:
        df[f'{col}_log1p'] = np.log1p(df[col].clip(lower=0))
    df['balance_abs_log1p'] = np.log1p(df['balance'].abs())
    df['pdays_log1p'] = np.log1p(df['pdays_clean'])
    
    df['contacts_total'] = df['campaign'] + df['previous']
    df['duration_per_campaign'] = df['duration'] / (df['campaign']+1)
    df['balance_per_age'] = df['balance'] / (df['age']+1)
    df['previous_per_campaign'] = df['previous'] / (df['campaign']+1)
    df['campaign_x_previous'] = df['campaign'] * df['previous']
    df['duration_x_campaign'] = df['duration'] * df['campaign']
    
    df['has_any_loan'] = ((h=='yes')|(l=='yes')).astype(np.int8)
    df['is_default'] = (d=='yes').astype(np.int8)
    df['was_contacted_before'] = (df['pdays']!=-1).astype(np.int8)
    df['is_cellular'] = (ct=='cellular').astype(np.int8)
    df['balance_signed_log1p'] = np.sign(df['balance']) * np.log1p(df['balance'].abs())
    df['balance_negative'] = (df['balance']<0).astype(np.int8)
    df['balance_nonpositive'] = (df['balance']<=0).astype(np.int8)
    
    ps = df['pdays'].replace(-1,999)
    df['age_bucket'] = pd.cut(df['age'],[0,25,35,45,55,65,120],labels=['<=25','26-35','36-45','46-55','56-65','65+']).astype(str)
    df['campaign_bucket'] = pd.cut(df['campaign'],[-1,1,2,4,9,np.inf],labels=['1','2','3-4','5-9','10+']).astype(str)
    df['previous_bucket'] = pd.cut(df['previous'],[-1,0,1,3,np.inf],labels=['0','1','2-3','4+']).astype(str)
    df['pdays_bucket'] = pd.cut(ps,[-1,7,30,90,365,np.inf],labels=['<=1w','8-30d','31-90d','91-365d','365d+']).astype(str)
    df.loc[df['pdays']==-1,'pdays_bucket'] = 'never'
    df['duration_bucket'] = pd.cut(df['duration'],[-1,60,120,240,480,np.inf],labels=['<=1m','1-2m','2-4m','4-8m','8m+']).astype(str)
    df['day_bucket'] = pd.cut(df['day'],[0,10,20,31],labels=['early','mid','late'],include_lowest=True).astype(str)
    
    # Base crosses
    df['job_education'] = j+'__'+e
    df['job_marital'] = j+'__'+ma
    df['contact_month'] = ct+'__'+m
    df['poutcome_month'] = po+'__'+m
    df['loan_default'] = l+'__'+d
    df['contact_day_bucket'] = ct+'__'+df['day_bucket']
    df['month_day_bucket'] = m+'__'+df['day_bucket']
    df['history_state'] = np.where(df['previous']>0, po+'__seen', 'no_previous')
    
    # STATE CROSSES (exp019 additions)
    df['age_bucket_contact'] = df['age_bucket']+'__'+ct
    df['age_bucket_history_state'] = df['age_bucket']+'__'+df['history_state']
    df['loan_housing'] = l+'__'+h
    df['contact_history_state'] = ct+'__'+df['history_state']
    df['month_pdays_bucket'] = m+'__'+df['pdays_bucket']
    df['job_contact'] = j+'__'+ct
    df['education_contact'] = e+'__'+ct
    df['housing_default'] = h+'__'+d
    df['marital_loan'] = ma+'__'+l
    
    cats = BASE_CAT + ['age_bucket','campaign_bucket','previous_bucket','pdays_bucket',
        'duration_bucket','day_bucket','job_education','job_marital','contact_month',
        'poutcome_month','loan_default','contact_day_bucket','month_day_bucket','history_state',
        'age_bucket_contact','age_bucket_history_state','loan_housing','contact_history_state',
        'month_pdays_bucket','job_contact','education_contact','housing_default','marital_loan']
    for c in cats: df[c] = df[c].fillna('missing').astype(str)
    return df, cats

print(f'Features with state_crosses defined')

In [ ]:
def train_cb(x_tr, x_te, y, cats, params, seeds):
    oof, test = np.zeros(len(x_tr)), np.zeros(len(x_te))
    for seed in seeds:
        s_oof, s_test = np.zeros(len(x_tr)), np.zeros(len(x_te))
        for fi, (ti, vi) in enumerate(StratifiedKFold(FOLDS, True, seed).split(x_tr, y), 1):
            m = CatBoostClassifier(**params, auto_class_weights='Balanced', random_seed=seed)
            m.fit(x_tr.iloc[ti].reset_index(drop=True), y.iloc[ti].reset_index(drop=True),
                  eval_set=(x_tr.iloc[vi].reset_index(drop=True), y.iloc[vi].reset_index(drop=True)),
                  cat_features=cats, use_best_model=True, early_stopping_rounds=250, verbose=False)
            s_oof[vi] = m.predict_proba(x_tr.iloc[vi].reset_index(drop=True))[:,1]
            s_test += m.predict_proba(x_te)[:,1] / FOLDS
        print(f'Seed {seed}: {roc_auc_score(y, s_oof):.6f}')
        oof += s_oof / len(seeds)
        test += s_test / len(seeds)
    return oof, test

train = pd.read_csv('/kaggle/input/fiicode-2026-ai-competition/train.csv')
test = pd.read_csv('/kaggle/input/fiicode-2026-ai-competition/test.csv')
y = train[TARGET].astype(int)
print(f'Train: {len(train)}, Test: {len(test)}')

x_train, cats = build_state_crosses(train.drop(columns=[TARGET]))
x_test, _ = build_state_crosses(test)
print(f'Features: {len(x_train.columns)}, Cats: {len(cats)}')

cb_oof, cb_test = train_cb(x_train, x_test, y, cats, CB_PARAMS, SEEDS)
print(f'\nFinal OOF AUC: {roc_auc_score(y, cb_oof):.6f}')

submission = pd.DataFrame({ID_COL: test[ID_COL], TARGET: np.clip(rank_norm(cb_test), 1e-6, 1-1e-6)})
submission.to_csv('submission.csv', index=False)
print('Saved submission.csv')
submission.head()